# Rolling Statistical Anomaly Detector

This notebook implements the second anomaly-detection approach for the CATS dataset: a rolling statistical detector.

Unlike the rule-based detector, which uses fixed thresholds derived from the nominal period, a rolling statistical detector evaluates observations relative to recent signal behaviour.

The detector will use rolling statistics calculated from historical observations to identify unusually large deviations in the telemetry signals.

The implementation will preserve temporal ordering and avoid using ground-truth anomaly labels or metadata to generate predictions.

The detector will be evaluated against the same CATS ground truth and compared with the rule-based baseline using observation-level and event-level metrics.

## Step 2 — Definition the Rolling Detection Rule

The rolling statistical detector will use a moving window of recent observations to estimate the local behaviour of each telemetry signal.

For each signal, the detector will calculate a rolling mean and rolling standard deviation using historical observations. An observation will be flagged when its value falls sufficiently far from the rolling mean relative to the rolling standard deviation.

The detector will use a minimum history requirement before generating predictions. This avoids making decisions from an insufficient number of observations.

To preserve temporal ordering, the rolling statistics will be shifted so that the observation being evaluated is not included in the statistics used to evaluate itself.

The window size and deviation threshold will be selected explicitly and documented before evaluation. Ground-truth anomaly labels and metadata will not be used to select them.

## Step 3 — Set the Initial Detector Parameters

The initial rolling statistical detector will use a 60-observation window, equivalent to 60 seconds because CATS is sampled at 1 Hz.

For each telemetry signal, the detector will calculate the rolling mean and rolling standard deviation from the preceding 60 observations. The statistics will be shifted by one observation so that the current observation is excluded from its own reference window.

An observation will be flagged when its value is more than three rolling standard deviations above or below the historical rolling mean.

A complete 60-observation history will be required before the detector generates a prediction.

These parameters establish an initial baseline. They are not assumed to be optimal, and their performance will be evaluated using the CATS ground truth after predictions have been generated.

## Step 4 — Load the Dataset and Define the Signal Inputs

This step loads the CATS dataset and defines the telemetry signals that the rolling statistical detector will evaluate.

The dataset contains 5,000,000 timestamped observations. The first 1,000,000 observations form the documented nominal period, followed by the evaluation period.

The detector will use only the 17 telemetry signals as inputs. The ground-truth anomaly label and category will be retained for later evaluation and excluded from the detector inputs.

The timestamp index will preserve the temporal order required for calculating rolling statistics.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cats" / "data.parquet"

df = pd.read_parquet(DATA_PATH)

signal_columns = [
    "aimp", "amud", "arnd", "asin1", "asin2",
    "adbr", "adfl", "bed1", "bed2", "bfo1",
    "bfo2", "bso1", "bso2", "bso3", "ced1",
    "cfo1", "cso1",
]

print("Dataset shape:", df.shape)
print("Timestamp index:", isinstance(df.index, pd.DatetimeIndex))
print("Signals defined:", len(signal_columns))
print("Signal columns present:", all(
    column in df.columns for column in signal_columns
))
print("Missing values in signal columns:", df[signal_columns].isna().sum().sum())

Dataset shape: (5000000, 19)
Timestamp index: True
Signals defined: 17
Signal columns present: True
Missing values in signal columns: 0


### Conclusion

The CATS dataset loaded successfully with 5,000,000 observations and a datetime index.

All 17 telemetry signals are present, and the original signal data contains no missing values.

The input definition explicitly excludes the ground-truth label and category from the detector features. This establishes the correct inputs for implementing the rolling statistical detector.

The dataset is ready for the next step: calculating rolling means and standard deviations using historical observations only.

## Step 5 — Calculate Historical Rolling Statistics

This step calculates a rolling mean and rolling standard deviation for each telemetry signal using the preceding 60 observations.

The rolling statistics will be shifted by one observation so that the current observation is not included in the reference values used to evaluate it.

A complete 60-observation history is required before the statistics are available. Predictions will not be generated in this step.

The resulting rolling statistics will be used in the next step to calculate deviations and generate anomaly predictions.

In [2]:
WINDOW_SIZE = 60

rolling_mean = (
    df[signal_columns]
    .rolling(window=WINDOW_SIZE, min_periods=WINDOW_SIZE)
    .mean()
    .shift(1)
)

rolling_std = (
    df[signal_columns]
    .rolling(window=WINDOW_SIZE, min_periods=WINDOW_SIZE)
    .std()
    .shift(1)
)

print("Rolling mean shape:", rolling_mean.shape)
print("Rolling standard deviation shape:", rolling_std.shape)
print("First observation with rolling statistics:", rolling_mean.dropna().index.min())
print("Missing rolling means:", rolling_mean.isna().sum().sum())
print("Missing rolling standard deviations:", rolling_std.isna().sum().sum())

Rolling mean shape: (5000000, 17)
Rolling standard deviation shape: (5000000, 17)
First observation with rolling statistics: 2023-01-01 00:01:00
Missing rolling means: 1020
Missing rolling standard deviations: 1020
